In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "openai_api_key"

In [ ]:
import os
from openai import OpenAI
import pandas as pd
from tqdm import tqdm
import time
import numpy as np

# Initialize OpenAI client
client = OpenAI()

# System prompt 
SYSTEM_INSTRUCTIONS = (
    "You are designed to answer users' questions about expectations for "
    "future inflation. Respond with only a numerical percentage estimate."
)

# Questions to ask
questions = [
    "What do you expect the rate of inflation to be over the next 12 months? Please give your best guess.",
    "What do you expect the rate of inflation to be over the 12-month period beginning 24 months from now and ending 36 months from now? Please give your best guess."
]

# Number of runs per question-treatment pair
num_runs = 100  # We can change this value to get more or fewer responses per condition

# Treatments including control
treatments = [
    # Control (no information)
    {
        "treatment": "T_0",
        "description": "Control with no information",
        "info": ""
    },
    # a. Changes in Language Complexity
    # Neutral Information
    {
        "treatment": "T_a1",
        "description": "Neutral - Simplified language",
        "info": (
            "The Federal Reserve monitors economic data like employment figures, prices, "
            "and economic growth to make decisions about interest rates. The current "
            "Federal Funds Rate is 4.25%-4.5%."
        )
    },
    {
        "treatment": "T_a2",
        "description": "Neutral - Technical language",
        "info": (
            "The Federal Open Market Committee utilizes a range of economic indicators "
            "including labor market conditions, inflation pressures, inflation expectations, "
            "and financial developments to calibrate monetary policy. The target range "
            "for the federal funds rate is currently 4.25 to 4.50 percent."
        )
    },
    # Anti-Recession (Accommodative Policy)
    {
        "treatment": "T_a3",
        "description": "Anti-recession - Simplified language",
        "info": (
            "The Federal Reserve is lowering interest rates to help boost the economy. "
            "This makes it cheaper for people and businesses to borrow money, which can "
            "create more jobs and economic activity."
        )
    },
    {
        "treatment": "T_a4",
        "description": "Anti-recession - Technical language",
        "info": (
            "The Federal Open Market Committee is implementing accommodative monetary policy "
            "by reducing the target range for the federal funds rate to stimulate aggregate demand. "
            "This policy adjustment is intended to facilitate credit accessibility, promote employment "
            "growth, and support economic expansion."
        )
    },
    # Anti-Inflation (Restrictive Policy)
    {
        "treatment": "T_a5",
        "description": "Anti-inflation - Simplified language",
        "info": (
            "The Federal Reserve is raising interest rates to help bring down high prices. "
            "Higher interest rates make borrowing more expensive, which slows down spending "
            "and helps control rising costs."
        )
    },
    {
        "treatment": "T_a6",
        "description": "Anti-inflation - Technical language",
        "info": (
            "The Federal Open Market Committee is implementing contractionary monetary policy "
            "by increasing the target range for the federal funds rate to counter inflationary pressures. "
            "This policy stance is designed to moderate demand, restore price stability, and anchor "
            "inflation expectations at levels consistent with the Committee's 2 percent objective."
        )
    },
    # b. Framing of Policy Commitments
    # Neutral Information
    {
        "treatment": "T_b1",
        "description": "Neutral - Conditional statement",
        "info": (
            "The Federal Reserve will adjust interest rates based on incoming economic data. "
            "Future policy decisions will depend on developments in employment, inflation, and "
            "broader economic conditions."
        )
    },
    {
        "treatment": "T_b2",
        "description": "Neutral - Unconditional statement",
        "info": (
            "The Federal Reserve will hold its next policy meeting on June 17-18. The committee "
            "will issue its regular statement and economic projections following the conclusion "
            "of the meeting."
        )
    },
    # Anti-Recession (Accommodative Policy)
    {
        "treatment": "T_b3",
        "description": "Anti-recession - Conditional statement",
        "info": (
            "The Federal Reserve will consider cutting interest rates if economic growth continues "
            "to slow and unemployment rises above 4.5%. The committee remains prepared to ease "
            "monetary policy if risks to economic activity increase."
        )
    },
    {
        "treatment": "T_b4",
        "description": "Anti-recession - Unconditional statement",
        "info": (
            "The Federal Reserve will reduce the federal funds rate by 0.25 percentage points at its "
            "next meeting and plans further cuts totaling 0.75 percentage points by the end of the year "
            "to support economic growth."
        )
    },
    # Anti-Inflation (Restrictive Policy)
    {
        "treatment": "T_b5",
        "description": "Anti-inflation - Conditional statement",
        "info": (
            "The Federal Reserve will consider additional rate increases if inflation remains elevated "
            "and fails to show substantial progress toward the 2% target. The committee stands ready "
            "to tighten policy further as warranted by the data."
        )
    },
    {
        "treatment": "T_b6",
        "description": "Anti-inflation - Unconditional statement",
        "info": (
            "The Federal Reserve will maintain high interest rates throughout 2025. The committee will "
            "not reduce rates until it has gained complete confidence that inflation is returning to the 2% "
            "target sustainably."
        )
    },
    # c. Time Horizons of Guidance
    # Neutral Information
    {
        "treatment": "T_c1",
        "description": "Neutral - Short-term horizon",
        "info": (
            "The Federal Reserve will assess incoming data over the next six weeks before its June meeting. "
            "The committee will consider the most recent inflation readings and employment report when "
            "making its next policy decision."
        )
    },
    {
        "treatment": "T_c2",
        "description": "Neutral - Long-term horizon",
        "info": (
            "The Federal Reserve's long-term goals include price stability and maximum sustainable employment. "
            "Over the coming years, the committee aims to conduct monetary policy that achieves inflation "
            "averaging 2% over time."
        )
    },
    # Anti-Recession (Accommodative Policy)
    {
        "treatment": "T_c3",
        "description": "Anti-recession - Short-term horizon",
        "info": (
            "The Federal Reserve is focused on improving economic conditions in the near term. In the next "
            "three months, the committee will prioritize actions that can quickly boost employment and "
            "economic activity."
        )
    },
    {
        "treatment": "T_c4",
        "description": "Anti-recession - Long-term horizon",
        "info": (
            "The Federal Reserve is launching a multi-year accommodative policy approach. The committee plans "
            "to maintain lower interest rates throughout the next two years to ensure a durable economic recovery "
            "and return to maximum employment."
        )
    },
    # Anti-Inflation (Restrictive Policy)
    {
        "treatment": "T_c5",
        "description": "Anti-inflation - Short-term horizon",
        "info": (
            "The Federal Reserve will maintain its current restrictive stance for the next three months. The committee "
            "expects to see meaningful progress on inflation reduction within this quarter before considering any policy adjustments."
        )
    },
    {
        "treatment": "T_c6",
        "description": "Anti-inflation - Long-term horizon",
        "info": (
            "The Federal Reserve is committed to a sustained campaign against inflation over the next several years. The committee "
            "anticipates that returning inflation to the 2% target will require maintaining restrictive policy well into 2026."
        )
    }
]

def run_experiment(treatment, question, run_id):
    # For control treatment, don't include the info part
    if treatment['info'] == "":
        prompt = f"Answer this question: {question}"
    else:
        prompt = f"Considering: {treatment['info']}  Answer this question: {question}"
    
    try:
        # Responses API expects 'instructions' and 'input' as strings
        resp = client.responses.create(
            model="gpt-4.1",
            instructions=SYSTEM_INSTRUCTIONS,
            input=prompt,
            temperature=1,
            top_p=1,
            max_output_tokens=10
        )
        text = resp.output_text.strip()
    except Exception as e:
        # fallback to Chat Completions if needed
        try:
            chat = client.chat.completions.create(
                model="gpt-4.1",
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTIONS},
                    {"role": "user",   "content": prompt}
                ],
                temperature=1,
                top_p=1,
                max_tokens=10
            )
            text = chat.choices[0].message.content.strip()
        except Exception as e2:
            print(f"Both endpoints failed for {treatment['treatment']} (run {run_id}): {e2}")
            text = "ERROR"
    
    return {
        "treatment_id": treatment["treatment"],
        "treatment_description": treatment["description"],
        "treatment_info": treatment["info"],
        "question": question,
        "run_id": run_id,
        "prompt": prompt,
        "raw_response": text,
        "response": text.rstrip("%").strip()
    }

# Run all experiments
def main():
    # Create output directory if it doesn't exist
    os.makedirs("results", exist_ok=True)
    
    results = []
    total = len(treatments) * len(questions) * num_runs
    print(f"Running {total} experiments ({num_runs} runs per condition)...")
    
    for treatment in tqdm(treatments, desc="Treatments"):
        for q_idx, q in enumerate(questions):
            for run in range(num_runs):
                time.sleep(1)  # avoid rate limits
                r = run_experiment(treatment, q, run+1)
                results.append(r)
    
    # Save detailed results
    df = pd.DataFrame(results)
    df.to_csv("results/detailed_inflation_expectations_results.csv", index=False)
    
    # Process for summary output (one row per treatment)
    summary_data = []
    
    for treatment in treatments:
        # Get all runs for this treatment
        treatment_df = df[df['treatment_id'] == treatment['treatment']]
        
        # Calculate average responses for each question
        q1_responses = treatment_df[treatment_df['question'] == questions[0]]['response']
        q2_responses = treatment_df[treatment_df['question'] == questions[1]]['response']
        
        # Convert to numeric, handling non-numeric responses
        q1_numeric = pd.to_numeric(q1_responses, errors='coerce')
        q2_numeric = pd.to_numeric(q2_responses, errors='coerce')
        
        # Calculate means and standard deviations
        q1_mean = q1_numeric.mean()
        q1_std = q1_numeric.std()
        q2_mean = q2_numeric.mean()
        q2_std = q2_numeric.std()
        
        # Get all individual responses for distribution analysis
        q1_all_responses = q1_responses.tolist()
        q2_all_responses = q2_responses.tolist()
        
        # Get sample prompt (first run)
        sample_prompt_short = treatment_df[treatment_df['question'] == questions[0]].iloc[0]['prompt'] if not treatment_df.empty else "N/A"
        sample_prompt_long = treatment_df[treatment_df['question'] == questions[1]].iloc[0]['prompt'] if not treatment_df.empty else "N/A"
        
        summary_data.append({
            'treatment_id': treatment['treatment'],
            'treatment_description': treatment['description'],
            'treatment_info': treatment['info'],
            'short_run_mean': q1_mean,
            'short_run_std': q1_std,
            'long_run_mean': q2_mean,
            'long_run_std': q2_std,
            'short_run_responses': q1_all_responses,
            'long_run_responses': q2_all_responses,
            'short_run_prompt': sample_prompt_short,
            'long_run_prompt': sample_prompt_long
        })
    
    # Create summary DataFrame and save
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv("inflation_expectations_summary.csv", index=False)
    
    print("All done! Results saved to:")
    print("  - inflation_expectations_summary.csv (main summary)")
    print("  - results/detailed_inflation_expectations_results.csv (detailed results)")

if __name__ == "__main__":
    main()